<a href="https://colab.research.google.com/github/GregKyri/Greg-Kyritsakas/blob/main/Main_Input_file.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Model for the selection of the treatment trains for a water treatment plant**  

>### **Give access to Google Drive**

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
%cd "/content/drive/My Drive/Colab Notebooks/WP6"

Mounted at /content/drive
/content/drive/My Drive/Colab Notebooks/WP6


>### **Calculate DWTP flow (including different patterns and needs)**

In [ ]:
%run "/content/drive/My Drive/Colab Notebooks/WP6/flowcalc.ipynb"
dailyflow, lanes, units, flowlane, flowunit = flowcalc()

Population numbers (comma-separated):5000,10000,15000
Water consumption per person per day (l/p/day): 150
Leakage estimation (0-1): 0.1
Are there any irrigation needs? (Yes/No): Yes
Are there any industrial needs? (Yes/No): Yes
Irrigation flow (m3/day):100
Industrial flow (m3/day):20
Set the number of different consumption periods: 2
Maximum water consumption per person per day (l/p/day): 250
Set the number of independent lanes: 2
Number of units per lane:2


>### **Key parameters and Water Source selection**

In [ ]:
%run "/content/drive/My Drive/Colab Notebooks/WP6/sourceoption.ipynb"
watersource, inputfilename = sourceoption()
Partable = pd.read_excel(inputfilename)
%run "/content/drive/My Drive/Colab Notebooks/WP6/inletparam.ipynb"
Partable2 = inletparam(watersource, Partable)
del inputfilename, Partable

Select one option:
1. Groundwater
2. Surface
3. Upland
4. River bank water
Enter the number corresponding to your choice:2
Enter the filename: HYLIKI Parameters averages 2013-2022.xlsx


>### **Import source for the Water Treatment Decision Tool**

In [ ]:
if watersource == "Groundwater":
    %run "/content/drive/My Drive/Colab Notebooks/WP6/groundwaterprocess.ipynb"
    treatmentprocess, treatalt = groundwaterprocess(watersource, Partable2)
elif watersource == "Surface":
    %run "/content/drive/My Drive/Colab Notebooks/WP6/surfacewaterprocess.ipynb"
    treatmentprocess, treatalt = surfacewaterprocess(watersource, Partable2)
elif watersource in ["Upland", "River bank water"]:
    %run "/content/drive/My Drive/Colab Notebooks/WP6/riverbankwaterprocess.ipynb"
    treatmentprocess, treatalt = riverbankwaterprocess(watersource, Partable2)

Is the algae concentration high?:
1. Yes
2. No
Enter your choice (1/2): 1
Use of an advanced technology (UF or MF)?:
1. Yes
2. No
Enter your choice (1/2): 2
The average maximum measured turbidity is 14.937999999999999
Is turbidity above 1 NTU?:
1. Yes
2. No
Enter your choice (1/2): 1
The average maximum measured DOC is 3.3732999999999995
NOM concentration in the water in (mg/l): 4
The average maximum measured hardness is 237.0
Is hardness in the water high?
1. Yes
2. No
Enter your choice (1/2): 1
Is disinfection required?
1. Yes
2. No
Enter your choice (1/2): 1
Treatment process: ['Screens', 'Microsieves', 'Coagulation', 'Flocculation', 'Floc Removal', 'Softening', 'Sand Filtration', 'OMP Removal', 'Disinfection']


>### **Treatment technologies and steps**

In [ ]:
%run "/content/drive/My Drive/Colab Notebooks/WP6/treatmentsteps.ipynb"
Treatment = treatmentsteps(treatmentprocess, treatalt)
print(Treatment)

Coagulation:
1. Cascade Mixing
2. Inline Mixing
3. Dynamic Mixing
Enter your choice (1/3): 1
Flocculation:
1. Static mixing
2. Dynamic Mixing
Enter your choice (1/2): 2
Floc Removal:
1. Sludge Blanket Clarification
2. Horizontal Settling
3. Titled-plate Settling
4. Dissolved Air Flotation
Enter your choice (1/4): 1
Softening:
1. Cationic IEX
2. Pellet Softening
3. NF/RO
Enter your choice (1/3):1
Sand Filtration:
1. Single Media Filtration
2. Dual Media Filtration
3. Moving Bed Filtration
Enter your choice (1/3):2
OMP Removal:
1. Granular Activated Carbon Filtration (GAC)
2. Powdered Activated Carbon Dosing (PAC)
3. MP-UV/H2O2
4. NF/RO
5. Ozonation
Enter your choice (1/5):2
Disinfection:
1. Ozonation
2. LP-UV
3. Chlorination
4. Chloramination
Enter your choice (1/4):3
Treatment technologies selected: ['Screens', 'Microsieves', 'Cascade Mixing', 'Dynamic mixing', 'Sludge Blanket Clarification', 'Cationic IEX', 'Dual Media Filtration', 'Powdered Activated Carbon Dosing (PAC)', 'Chlorinati

<ipython-input-33-06c6056609d7>:42: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  Treatment = Treatment.append(temp_row,ignore_index=True)


>### **Coaggulation calculations**

In [ ]:
# Check if "Rapid sand filtration" is in the bestTreatment list
if (Treatment['Treatment Steps'] == "Coagulation").any():
    # Call the rapidsandfilter function with appropriate arguments
    Dtheta, Coagvolume, Coagmixpower, CoagTanksize = Coagfunc(Partable2, dailyflow, Treatment)


>### **Rapid sand filtration calculations**

In [ ]:
# Check if "Sand filtration" is in the 'Treatment Steps' column of the DataFrame
if (Treatment['Treatment Steps'].str.lower().str.strip() == "sand filtration").any():
    # Call the rapidsandfilter function with appropriate arguments
    %run "/content/drive/My Drive/Colab Notebooks/WP6/rapidsandfilter.ipynb"
    filterArea, filterAreaperlane, filterWidthunit, filterLengthunit = rapidsandfilter(lanes,units,Partable2,dailyflow,Treatment)

Footprint per unit (m2) 70.95
Footprint of Sand Filtration(m2) 141.9
Total number of units 4
Total number of lanes 2
Sand filtration Length 23.650000000000002
Sand filtration width 70.95


>### **Create the WTWs flowchart**

In [ ]:
%run "/content/drive/My Drive/Colab Notebooks/WP6/finalflowchart.ipynb"
flowchart= finalflowchart(Treatment)